In [6]:
import os
import random
import getpass
import warnings
import collections
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR100, FashionMNIST
from torch.utils.data import DataLoader

In [7]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [8]:
DATA_ROOT   = Path("./data")
FIGURES_DIR = Path("./figures")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")

[INFO] Device: cpu


In [10]:
BATCH_SIZE   = 128
NUM_WORKERS  = 2
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

In [12]:
def pixel_stats(dataset, n_samples: int = 2000, label: str = ""):
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    stack   = np.stack([dataset[i][0].numpy() for i in indices])   
    if stack.ndim == 3:                                             
        stack = stack[:, np.newaxis]
    mean = stack.mean(axis=(0, 2, 3))
    std  = stack.std(axis=(0, 2, 3))
    print(f"  [{label}] mean={np.round(mean,4)}  std={np.round(std,4)}")
    return mean, std

In [13]:
def show_sample_grid(dataset, class_names: list, n_classes: int = 10,
                     n_cols: int = 5, title: str = "", fname: str = "grid"):
    label2idx = collections.defaultdict(list)
    for idx, lbl in enumerate(get_labels(dataset)):
        label2idx[lbl].append(idx)

    chosen = random.sample(range(len(class_names)), min(n_classes, len(class_names)))
    fig, axes = plt.subplots(len(chosen), n_cols,
                             figsize=(n_cols * 1.6, len(chosen) * 1.6))
    if len(chosen) == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=12, fontweight="bold")

    for row, cls in enumerate(chosen):
        idxs = random.sample(label2idx[cls], min(n_cols, len(label2idx[cls])))
        for col, idx in enumerate(idxs):
            t   = dataset[idx][0]
            img = t.permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            ax  = axes[row][col]
            ax.imshow(img if img.shape[2] == 3 else img[:, :, 0], cmap="gray")
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(class_names[cls], fontsize=7,
                              rotation=0, labelpad=55, va="center")
    plt.tight_layout()
    save_fig(fname)
    plt.show()